# Stim circuits for the Steane syndrome-extraction comparison

Draws the stim circuits that are sampled for Table `tab:goto-resources`, one syndrome-extraction half-cycle each, in both bases:

- **Goto baseline**: the search input, after `swap_basis`
- **Searched (29)**: the race winner, Bell reductions (7,8), (10,8), (10,9), (10,8) in the paper's convention (see below), relabelled by `normalize_circuit` exactly as the simulation sees it
- **Po'or FT / recovery**: the dynamic optimized Steane primary circuit and its non-FT fallback (`low/simulations/dynamic_floqetified_stean.py`)

Run from the repository root.

In [ ]:
import sys
from pathlib import Path

from IPython.display import display, Markdown

ROOT = Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "low" / "simulations")]

import bell_reduction
import circuit
import floqetified_steane_custom as fsc
import dynamic_floqetified_stean as poor
from sweep import build_reductions, normalize_circuit

Display options. With `SHOW_NOISE = False` the noise channels (`DEPOLARIZE2`, idle `Z_ERROR`, SPAM) are stripped so the gates are readable. `"timeslice-svg"` draws one panel per stim `TICK`; these circuits carry no `TICK`s, so `"timeline-svg"` is the useful one.

In [ ]:
SHOW_NOISE = False
DIAGRAM = "timeline-svg"
WINNER = 29

In [ ]:
def stim_pair(gates):
    """Z- and X-basis stim circuits for a gate list, built as FlaggedSE builds them."""
    h1, cnots, h2 = fsc.split_h_layers(gates)
    return {b: fsc.floq_steane_FT_SE(b, h1, cnots, h2) for b in "ZX"}


baseline = bell_reduction.swap_basis(
    [g for g in circuit.generate_goto_circuit() if g.name != "barrier"])
searched, _ = normalize_circuit(build_reductions("goto")[WINNER])

CIRCUITS = {
    "Goto baseline": stim_pair(baseline),
    f"Searched ({WINNER})": stim_pair(searched),
    "Po'or FT": {"Z": poor.FLAGGED_Z_SYNDROME_MEASUREMENT,
                 "X": poor.FLAGGED_X_SYNDROME_MEASUREMENT},
    "Po'or recovery": {"Z": poor.UNFLAGGED_Z_SYNDROME_MEASUREMENT,
                       "X": poor.UNFLAGGED_X_SYNDROME_MEASUREMENT},
}

In [ ]:
def show(name, basis):
    c = CIRCUITS[name][basis]
    if not SHOW_NOISE:
        c = c.without_noise()
    display(Markdown(f"### {name}, {basis} basis  ({c.num_qubits} qubits)"))
    display(c.diagram(DIAGRAM))


for name in CIRCUITS:
    for basis in "ZX":
        show(name, basis)

## As Qiskit circuits

`stim_to_qiskit` drops every noise channel (`without_noise()`) and maps the remaining instructions one to one. The wires are grouped as `gates_to_qiskit` groups them: data qubits 0–6 in register `d`, the ancillas the circuit touches in register `a`. `TICK`s are dropped rather than drawn as barriers. Each measurement gets its own classical bit, in stim's record order, and `MR` becomes a measure followed by a reset.

In [ ]:
import qiskit
from qiskit.transpiler.passes import RemoveBarriers

N_DATA = 7


def stim_to_qiskit(c, n_data=N_DATA):
    """A noiseless QuantumCircuit for a stim circuit made of H, CX, resets and measurements."""
    c = c.without_noise().flattened()
    used = sorted({t.value for inst in c for t in inst.targets_copy() if t.is_qubit_target})
    anc = [q for q in used if q >= n_data]
    dr = qiskit.QuantumRegister(n_data, "d")
    ar = qiskit.QuantumRegister(len(anc), "a")
    cr = qiskit.ClassicalRegister(c.num_measurements, "m")
    qc = qiskit.QuantumCircuit(dr, ar, cr)
    wire = {q: dr[q] for q in range(n_data)} | {q: ar[i] for i, q in enumerate(anc)}

    m = 0
    for q in range(len(anc)):
        qc.reset(q+n_data)
    for inst in c:
        name = inst.name
        qs = [wire[t.value] for t in inst.targets_copy()]
        if name == "TICK":
            continue
        elif name == "H":
            for q in qs:
                qc.h(q)
        elif name in ("CX", "CNOT"):
            for ctl, tgt in zip(qs[::2], qs[1::2]):
                qc.cx(ctl, tgt)
        elif name in ("R", "RX"):
            for q in qs:
                qc.reset(q)
                if name == "RX":
                    qc.h(q)
        elif name in ("M", "MR", "MX", "MRX"):
            for q in qs:
                if name in ("MX", "MRX"):
                    qc.h(q)
                qc.measure(q, cr[m])
                m += 1
                if name in ("MR", "MRX"):
                    if name == "MRX":
                        qc.h(q)
        else:
            raise ValueError(f"no qiskit translation for stim instruction {name}")
    return qc


QISKIT = {name: {b: stim_to_qiskit(c) for b, c in d.items()} for name, d in CIRCUITS.items()}

In [ ]:
def show_qiskit(name, basis, fold=-1):
    qc = QISKIT[name][basis]
    cx_depth = qc.depth(lambda i: i.operation.name == "cx")
    display(Markdown(f"### {name}, {basis} basis  "
                     f"({qc.count_ops().get('cx', 0)} CNOTs, CNOT depth {cx_depth})"))
    display(qc.draw("mpl", fold=fold))


for name in QISKIT:
    for basis in "ZX":
        show_qiskit(name, basis)

## Searched circuit, step by step

The winner's derivation replayed from the search input; the five stages are those of `paper/circuit/der_steane`. Each step is one `bell_reduction.reduce(circuit, q1, q2)`, which keeps wire `q1`, transposes its gates and appends `q2`'s, so every step removes one ancilla and one CNOT.

Headings follow the convention of the paper's rewrite-rules section: the Bell reduction of $(q_2, q_1)$ keeps $q_2$, the $|0\rangle$-initialized wire whose gates are transposed, and removes $q_1$. When the code keeps the $|+\rangle$ side instead (step 3), the composite reduction includes a basis swap that makes it the $|0\rangle$ side, so the pair is still written survivor first. Labels are those of the circuit the reduction is applied to, with the surviving ancillas numbered consecutively from 7, as in the figures and as `circuit.gates_to_qiskit` draws them (wire `7 + i` is `a_i`). The search's own path, (7,8), (11,9), (12,11), (13,10), leaves gaps where wires were removed; `winner.reduction_path` keeps those labels, since that is what `reduce` takes. The last step is checked against `build_reductions("goto")[WINNER]`.

In [ ]:
winner = build_reductions("goto")[WINNER]


def drawn_pair(gates, q1, q2):
    """(q1, q2) in the drawing's numbering: ancilla wires renumbered 7, 8, ... in order."""
    anc = sorted({q for g in gates for q in g.qubits if q >= N_DATA})
    pos = {q: N_DATA + i for i, q in enumerate(anc)}
    return pos[q1], pos[q2]


steps = [("search input (Goto, basis-swapped)", baseline)]
current = baseline
for q1, q2 in winner.reduction_path:
    a, b = drawn_pair(current, q1, q2)
    current = bell_reduction.reduce(current, q1, q2)
    steps.append((f"after Bell reduction ({a}, {b})", current))

assert steps[-1][1] == winner.circuit, "replay does not reach the searched circuit"

# gates_to_qiskit puts a barrier before the measurements; strip it here, not there,
# since draw_goto_circuit.py relies on it for the paper figures.
STEPS_QISKIT = [(label, RemoveBarriers()(circuit.gates_to_qiskit(gates))) for label, gates in steps]

In [ ]:
for k, (label, qc) in enumerate(STEPS_QISKIT):
    cx_depth = qc.depth(lambda i: i.operation.name == "cx")
    display(Markdown(f"### Step {k}: {label}  "
                     f"({qc.num_qubits - N_DATA} ancillas, {qc.count_ops().get('cx', 0)} CNOTs, "
                     f"CNOT depth {cx_depth})"))
    display(qc.draw("mpl", fold=-1))

The stim text of any one circuit, for checking against the diagram:

In [ ]:
print(CIRCUITS[f"Searched ({WINNER})"]["Z"].without_noise())

In [ ]:
import qiskit2qunatikz

In [ ]:
QISKIT['Goto baseline']['X'].draw('mpl', scale=0.5)

In [ ]:
qiskit2qunatikz.convert(QISKIT['Goto baseline']['X'])

In [ ]:
qc = qiskit.QuantumCircuit()